In [ ]:
cell_start(0, "Setup e Autenticacao")import os, sys, json, subprocess, time, io, gc, shutil, glob, base64from google.oauth2.credentials import Credentialsfrom google.auth.transport.requests import Requestfrom googleapiclient.discovery import buildfrom googleapiclient.http import MediaFileUpload, MediaIoBaseDownloadimport requests as http_requestsNOTEBOOK_NAME = "renderizador-kaggle-pt-2"STEP_NAME = "step_render_pt2"print("Instalando FFmpeg...")os.system("apt-get install -y ffmpeg > /dev/null 2>&1")def _load_secrets():    try:        from kaggle_secrets import UserSecretsClient        _s = UserSecretsClient()        def _get(name):            try: return _s.get_secret(name)            except: return ""        print("Carregando Kaggle Secrets...")        return _get    except ImportError:        from dotenv import load_dotenv; load_dotenv()        return lambda name: os.getenv(name, "")_get = _load_secrets()DRIVE_REFRESH_TOKEN = _get("DRIVE_REFRESH_TOKEN")DRIVE_CLIENT_ID = _get("DRIVE_CLIENT_ID")DRIVE_CLIENT_SECRET = _get("DRIVE_CLIENT_SECRET")DATABASE_URL = _get("DATABASE_URL")PROJECT_ID = _get("PIPELINE_PROJECT_ID")print("Autenticando Drive...")try:    _creds = Credentials(token=None, refresh_token=DRIVE_REFRESH_TOKEN,        token_uri="https://oauth2.googleapis.com/token",        client_id=DRIVE_CLIENT_ID, client_secret=DRIVE_CLIENT_SECRET,        scopes=["https://www.googleapis.com/auth/drive"])    _creds.refresh(Request())    drive_service = build("drive", "v3", credentials=_creds)    print("Drive OK!")except Exception as e:    drive_service = None    print(f"Drive falhou: {e}")def _buscar_id(caminho):    partes = caminho.strip("/").split("/")    pid = "root"    for p in partes:        q = f"name=\'{p}\' and \'{pid}\' in parents and trashed=false"        r = drive_service.files().list(q=q, fields="files(id,mimeType)").execute()        a = r.get("files", [])        if not a: return None        pid = a[0]["id"]    return piddef _garantir_pasta(caminho):    partes = caminho.strip("/").split("/")    pid = "root"    for p in partes:        q = f"name=\'{p}\' and \'{pid}\' in parents and trashed=false and mimeType=\'application/vnd.google-apps.folder\'"        r = drive_service.files().list(q=q, fields="files(id)").execute()        e = r.get("files", [])        if e: pid = e[0]["id"]        else:            nova = drive_service.files().create(body={"name": p, "mimeType": "application/vnd.google-apps.folder", "parents": [pid]}, fields="id").execute()            pid = nova["id"]    return piddef baixar_do_drive(caminho_drive, destino_local):    if os.path.exists(destino_local): return True    try:        fid = _buscar_id(caminho_drive)        if not fid: print(f"  Nao encontrado: {caminho_drive}"); return False        req = drive_service.files().get_media(fileId=fid)        os.makedirs(os.path.dirname(destino_local) or ".", exist_ok=True)        with open(destino_local, "wb") as fh:            dl = MediaIoBaseDownload(fh, req); done = False            while not done: _, done = dl.next_chunk()        print(f"  Baixado: {caminho_drive}"); return True    except Exception as ex: print(f"  Erro: {ex}"); return Falsedef salvar_no_drive(caminho_local, caminho_drive):    if not drive_service or not os.path.exists(caminho_local): return    try:        partes = caminho_drive.strip("/").split("/")        nome = partes[-1]; pasta = "/".join(partes[:-1]) if len(partes) > 1 else ""        pid = _garantir_pasta(pasta) if pasta else "root"        q = f"name=\'{nome}\' and \'{pid}\' in parents and trashed=false"        r = drive_service.files().list(q=q, fields="files(id)").execute()        e = r.get("files", []); media = MediaFileUpload(caminho_local, resumable=True)        if e: drive_service.files().update(fileId=e[0]["id"], media_body=media).execute()        else: drive_service.files().create(body={"name": nome, "parents": [pid]}, media_body=media, fields="id").execute()        print(f"  Salvo: {caminho_drive}")    except Exception as ex: print(f"  Erro salvar: {ex}")_cell_timers = {}def _db_exec(query, params):    if not DATABASE_URL: return    try:        import psycopg2; conn = psycopg2.connect(DATABASE_URL); cur = conn.cursor()        cur.execute(query, params); conn.commit(); cur.close(); conn.close()    except: passdef cell_start(idx, name=""):    _cell_timers[idx] = time.time()    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")    if not PROJECT_ID: return    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,\'running\',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))    _db_exec("UPDATE pipeline_cell_tracking SET status=\'running\',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))def cell_end(idx, status="done", msg=""):    elapsed = ""    if idx in _cell_timers:        s = int(time.time() - _cell_timers.pop(idx))        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"    icon = "OK" if status == "done" else "ERRO"    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'_'*50}")    if not PROJECT_ID: return    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))def report_step(status, msg=""):    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")    if not PROJECT_ID: return    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))DRIVE_ENHANCER = "KAGGLE/PIPELINE/ENHANCER"DRIVE_OMNI = "KAGGLE/PIPELINE/OMNI"DRIVE_RENDER = "KAGGLE/PIPELINE/RENDER"BASE_PATH = "/kaggle/working"os.makedirs(BASE_PATH, exist_ok=True)cell_end(0, "done", "Setup concluido")

In [ ]:
cell_start(1, "Download Arquivos do Drive")# Baixar video enhanced + config + legendas + audio do Drivebaixar_do_drive(f"{DRIVE_ENHANCER}/pt2_enhanced.mp4", f"{BASE_PATH}/pt2_enhanced.mp4")baixar_do_drive(f"{DRIVE_OMNI}/videorender-project.json", f"{BASE_PATH}/videorender-project.json")baixar_do_drive(f"{DRIVE_OMNI}/legendas.ass", f"{BASE_PATH}/legendas.ass")baixar_do_drive(f"{DRIVE_OMNI}/audio_dublado.mp3", f"{BASE_PATH}/audio_dublado.mp3")# Carregar configimport shutilshutil.copy2(f"{BASE_PATH}/legendas.ass", f"{BASE_PATH}/legendas_temp.ass")with open(f"{BASE_PATH}/videorender-project.json", "r", encoding="utf-8") as f:    config = json.load(f)VIDEO_INPUT = f"{BASE_PATH}/pt2_enhanced.mp4"AUDIO_INPUT = f"{BASE_PATH}/audio_dublado.mp3"ASS_LOCAL = f"{BASE_PATH}/legendas_temp.ass"OUTPUT_FILE = f"{BASE_PATH}/pt2_renderizado.mp4"print("Arquivos prontos!")print(f"  Video: {VIDEO_INPUT}")print(f"  Audio: {AUDIO_INPUT}")print(f"  ASS:   {ASS_LOCAL}")cell_end(1, "done", "Download Arquivos do Drive concluido")

In [ ]:
cell_start(2, "Build Comando FFmpeg")def build_ffmpeg_command(config, video_in, audio_in, ass_in, out_file):    filters = []    last_stream = "[0:v]"    overlay_inputs = []        out_format = config["video"].get("outputFormat", "16:9")    out_w, out_h = (1080, 1920) if out_format == "9:16" else (1920, 1080)        crop_info = config.get("cropZoom", {})    if crop_info.get("enabled"):        zs = crop_info.get("zoomStart", 1.0)        fx = crop_info.get("focusX", 0.5)        fy = crop_info.get("focusY", 0.5)        in_w = config["video"]["info"]["width"]        in_h = config["video"]["info"]["height"]        cw = int(in_w / zs)        ch = int(in_h / zs)        cx = int((in_w - cw) * fx)        cy = int((in_h - ch) * fy)        filters.append(f"{last_stream}crop={cw}:{ch}:{cx}:{cy},scale={out_w}:{out_h}[cropped]")        last_stream = "[cropped]"    else:        filters.append(f"{last_stream}scale={out_w}:{out_h}:force_original_aspect_ratio=increase,crop={out_w}:{out_h}[scaled]")        last_stream = "[scaled]"        cg = config.get("colorGrade", {})    if cg:        b = cg.get("brightness", 0) / 100.0        c = 1.0 + cg.get("contrast", 0) / 100.0        s = 1.0 + cg.get("saturation", 0) / 100.0        g = cg.get("gamma", 1.0)        filters.append(f"{last_stream}eq=brightness={b}:contrast={c}:saturation={s}:gamma={g}[colorgraded]")        last_stream = "[colorgraded]"        temp = cg.get("temperature", 0)        if temp != 0:            red_mod = temp / 100.0 if temp > 0 else 0            blue_mod = abs(temp) / 100.0 if temp < 0 else 0            filters.append(f"{last_stream}colorbalance=rm={red_mod}:bm={blue_mod}[temp_applied]")            last_stream = "[temp_applied]"        sharp = cg.get("sharpness", 1.0)        if sharp > 1.0:            amount = sharp - 1.0            filters.append(f"{last_stream}unsharp=5:5:{amount}:5:5:0.0[sharpened]")            last_stream = "[sharpened]"        v = cg.get("vignette", 0)        if v > 0:            filters.append(f"{last_stream}vignette=a={v}[vignetted]")            last_stream = "[vignetted]"    overlays = config.get("overlays", [])    for i, ov in enumerate(overlays):        if ov["type"] in ["image", "watermark"]:            header, encoded = ov["content"].split(",", 1)            ext = header.split(";")[0].split("/")[1]            filename = f"/kaggle/working/temp_overlay_{i}.{ext}"            with open(filename, "wb") as f:                f.write(base64.b64decode(encoded))            overlay_inputs.append(filename)            input_idx = len(overlay_inputs) + 1            ox = int((ov["x"] / 100.0) * out_w)            oy = int((ov["y"] / 100.0) * out_h)            ow = int((ov["width"] / 100.0) * out_w)            oh = int((ov["height"] / 100.0) * out_h)            opacity = ov.get("opacity", 1.0)            filters.append(f"[{input_idx}:v]scale={ow}:{oh}[ov_scaled_{i}]")            alpha_filter = f",colorchannelmixer=aa={opacity}" if opacity < 1.0 else ""            filters.append(f"[ov_scaled_{i}]format=rgba{alpha_filter}[ov_alpha_{i}]")            time_filter = ""            tin = ov.get("timeIn", 0)            tout = ov.get("timeOut", 0)            if tin > 0 or tout > 0:                if tout == 0: tout = 999999                time_filter = f":enable=\'between(t,{tin},{tout})\'"            filters.append(f"{last_stream}[ov_alpha_{i}]overlay=x={ox}:y={oy}{time_filter}[with_ov_{i}]")            last_stream = f"[with_ov_{i}]"        elif ov["type"] == "text":            ox = int((ov["x"] / 100.0) * out_w)            oy = int((ov["y"] / 100.0) * out_h)            txt = ov["content"].replace("'", "\\\'").replace(":", "\\\\:")            fsize = int((ov.get("fontSize", 32) * (out_h / 1080)))            fcolor = ov.get("fontColor", "#ffffff")            time_filter = ""            tin = ov.get("timeIn", 0)            tout = ov.get("timeOut", 0)            if tin > 0 or tout > 0:                if tout == 0: tout = 999999                time_filter = f":enable=\'between(t,{tin},{tout})\'"            filters.append(f"{last_stream}drawtext=text=\'{txt}\':x={ox}:y={oy}:fontsize={fsize}:fontcolor={fcolor}{time_filter}[with_txt_{i}]")            last_stream = f"[with_txt_{i}]"    filters.append(f"{last_stream}ass=\'legendas_temp.ass\'[subbed]")    last_stream = "[subbed]"        filter_complex = ";".join(filters)    cmd = ["ffmpeg", "-y", "-threads", "4", "-filter_threads", "4", "-i", video_in, "-i", audio_in]    for ov_file in overlay_inputs:        cmd.extend(["-i", ov_file])    cmd.extend(["-filter_complex", filter_complex, "-map", last_stream, "-map", "1:a",        "-c:v", "h264_nvenc", "-cq", "18", "-preset", "p6", "-c:a", "aac", "-b:a", "192k", "-shortest", out_file])    return cmdos.chdir(BASE_PATH)command = build_ffmpeg_command(config, VIDEO_INPUT, AUDIO_INPUT, ASS_LOCAL, OUTPUT_FILE)print("Comando FFmpeg gerado!")cell_end(2, "done", "Build Comando FFmpeg concluido")

In [ ]:
cell_start(3, "Renderizacao")print("Iniciando renderizacao...")process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)for line in process.stderr:    if "frame=" in line:        print(line.strip(), end="\r")process.wait()if process.returncode == 0:    print("\n\nRenderizacao concluida!")else:    print("\n\nErro na renderizacao!")    print(process.stderr.read() if hasattr(process.stderr, "read") else "")cell_end(3, "done", "Renderizacao concluido")

In [ ]:
cell_start(4, "Upload Resultado")salvar_no_drive(OUTPUT_FILE, f"{DRIVE_RENDER}/pt2_renderizado.mp4")report_step("done", "Renderizacao PT2 concluida")print("Upload concluido!")cell_end(4, "done", "Upload Resultado concluido")